# Local YOLOv8 training and frame inference

Install the YOLO extra and explicitly fetch assets before execution. COCO8 is only four train/four validation images, drawn from COCO train2017: pretrained overlap is possible. This notebook verifies execution, not generalization or traffic safety.

In [ ]:
import os
from pathlib import Path
import uuid
project = Path.cwd()
if not (project/'configs').is_dir():
    project = project.parent
assert (project/'configs').is_dir(), 'Launch from the project or notebooks directory'
os.chdir(project)
run = project/'artifacts'/('notebook-'+uuid.uuid4().hex[:12])

assets = Path(os.environ.get('UNET_YOLO_ASSETS',str(project/'assets/downloaded'))).resolve()
dataset = Path(os.environ.get('UNET_YOLO_DATA',str(assets/'coco8'))).resolve()
base = assets/'yolov8n.pt'
video = assets/'traffic_video.mp4'
assert base.is_file() and video.is_file() and (dataset/'images/val').is_dir(), 'Retrieve assets explicitly first'

In [ ]:
from unet_yolo.detection import train,predict
training = train(base,dataset,project/'configs/yolo-smoke.json',run/'training')
print(training)

## Reload the selected checkpoint for image inference

The chosen image is from validation, not an independent test.

In [ ]:
best = run/'training/run/weights/best.pt'
image_result = predict(best,dataset/'images/val/000000000042.jpg',run/'image')
from IPython.display import display
from PIL import Image
display(Image.open(run/'image/frame-0000.jpg'))
print(image_result)

## Bounded video smoke with the official base

The video uses the original pretrained base, not the four-image fine-tune. Only eight frames are processed here. Empty detections are a valid execution outcome and say nothing about whether traffic is present.

In [ ]:
video_result = predict(base,video,run/'video',max_frames=8)
assert video_result['frames_processed'] == 8
assert len((run/'video/detections.jsonl').read_text().splitlines()) == 8
display(Image.open(run/'video/frame-0000.jpg'))
print(video_result)

Full weights, source and annotated video remain local. Traffic preview source: Intel IoT DevKit sample-videos/car-detection.mp4 (CC BY 4.0); see ../docs/provenance.md. Original-coordinate boxes and source/checkpoint hashes are recorded in JSONL/JSON. No video ground-truth metric or production deployment is claimed.